<a href="https://colab.research.google.com/github/Ace2932/Fennec/blob/main/sim/nova_mjx/colab/fennec_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fennec — terrain-relative reward PROBE (Colab GPU)

Runs the probe for the terrain-relative reward fix (PR #130): resume the stairs teacher under the
corrected reward and ask *does it now step UP stairs instead of scraping the riser, without losing
the flat gait?*

**Flow:** config → get code (fresh clone, asserts the fix) → deps → drive → sanity → dry-run →
probe → judge (rollouts + `climbed`) → export.

Checkpoints live on Drive, so a disconnect costs nothing — re-run the probe cell and it resumes
its own checkpoint (does **not** re-graft from the teacher).

> Runtime → Change runtime type → **GPU** (T4 is enough) before starting.
> The kernel stays jax-free (jax runs only in `!python` subprocesses) so the rollout/rebuild cells
> can't hit the `os.fork()`+JAX deadlock.

## 1. Config — single source of truth (edit here, nowhere else)

In [2]:
# Every path/knob lives here; every cell reads from these. Re-run this cell FIRST after any
# 'Restart session' (a restart wipes these vars and the working dir).
BRANCH   = "main"          # after PR #130 merges. Before merge: "sim/terrain-relative-reward"
DRIVE    = "/content/drive/MyDrive"
TEACHER  = f"{DRIVE}/nova_policy_stairs_final.pkl"   # obs-226 stairs teacher this probe resumes
CKPT     = f"{DRIVE}/nova_stairs_fix"                # FRESH dir — never reuse nova_stairs_curr (old reward)
POLICY   = f"{DRIVE}/nova_policy_stairs_fix.pkl"     # flat pkl written atomically every eval
TIMESTEPS = 25_000_000                               # ~1.3 h on a T4
print("branch", BRANCH, "| ckpt", CKPT)

branch main | ckpt /content/drive/MyDrive/nova_stairs_fix


## 2. GPU + Python check

In [3]:
import sys
print("Python", sys.version.split()[0])   # want 3.11/3.12
!nvidia-smi -L || echo "NO GPU -> Runtime > Change runtime type > GPU"

Python 3.12.13
GPU 0: Tesla T4 (UUID: GPU-ce574f19-9691-20aa-a9df-8a3681c8a89c)


## 3. Get the code (fresh clone) — and REFUSE to run stale code

Fresh `rm -rf` + clone of the exact `BRANCH` — cannot run leftover code from a prior branch, and
prints the SHA + **asserts `_terrain_ground_z` is present**. If the fix is missing it raises rather
than waste a GPU-hour (the "Already up to date -> ran stale code" trap).

In [4]:
import subprocess
%cd /content
!rm -rf LE_NOVA
!git clone --depth 1 -b {BRANCH} https://github.com/Ace2932/LE_NOVA.git
%cd /content/LE_NOVA/sim/nova_mjx
sha = subprocess.check_output(["git","rev-parse","--short","HEAD"], text=True).strip()
has_fix = "_terrain_ground_z" in open("env.py").read()
print("HEAD", sha, "| terrain-relative fix present:", has_fix)
assert has_fix, ("env.py has no _terrain_ground_z -> PRE-fix code. Merge PR #130 (or set "
                 "BRANCH='sim/terrain-relative-reward' in cell 1) and re-run cells 1+3.")

/content
Cloning into 'LE_NOVA'...
remote: Enumerating objects: 686, done.
remote: Counting objects: 100% (686/686), done.
remote: Compressing objects: 100% (595/595), done.
remote: Total 686 (delta 104), reused 483 (delta 66), pack-reused 0 (from 0)
Receiving objects: 100% (686/686), 33.08 MiB | 13.67 MiB/s, done.
Resolving deltas: 100% (104/104), done.
/content/LE_NOVA/sim/nova_mjx
HEAD 5541353 | terrain-relative fix present: True


## 4. Install pinned deps
brax 0.14.2 + jax 0.6.0 is the validated window. **If the sanity cell reports `cpu`:**
`Runtime -> Restart session`, then re-run **from cell 1** (config + working dir are wiped by a
restart; the clone on disk persists so cell 3 is fast).

In [5]:
!pip install -q "jax[cuda12]==0.6.0" brax==0.14.2 "orbax-checkpoint>=0.11.22" \
    "mujoco>=3.10" "mujoco-mjx>=3.10" "imageio>=2.31" imageio-ffmpeg 2>&1 | tail -3
print("deps installed — if the next cell says cpu, restart session + re-run from cell 1")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 44.0 MB/s eta 0:00:00
deps installed — if the next cell says cpu, restart session + re-run from cell 1


## 5. Mount Drive

In [6]:
from google.colab import drive
import os
drive.mount("/content/drive")
assert os.path.exists(TEACHER), f"teacher pkl missing: {TEACHER}"
print("teacher:", TEACHER)

Mounted at /content/drive
teacher: /content/drive/MyDrive/nova_policy_stairs_final.pkl


## 6. Sanity — GPU backend (jax checked in a subprocess, not the kernel)

The `!python -c` keeps jax out of the notebook kernel so later `subprocess`/`!python` cells are
fork-safe. The commented line runs the fix's own suite (T1-T9 + mj_ray oracle) on this runtime —
~3-5 min, a one-time confidence check; uncomment on first use of a branch, skip on re-runs.

In [7]:
!python -c "import jax; b=jax.default_backend(); print('jax backend:', b); assert b=='gpu', 'not on GPU — restart session + re-run from cell 1'"
# !python test_terrain_relative.py 2>&1 | tail -3

jax backend: gpu


## 7. Probe — DRY RUN first

Prints the **reward fingerprint** — terrain 1.00, flat-frac 0.25, the graft source, and the git
SHA (with `(+uncommitted changes)` flag) — then exits without training. Confirm it's the fixed code
and the right config before spending GPU.

In [7]:
!python train.py --heightmap --terrain 1.0 --stair-frac 0.6 --flat-frac 0.25 \
    --restore-params-pkl {TEACHER} --ckpt {CKPT} --out {POLICY} \
    --timesteps {TIMESTEPS} --dry-run

Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'
/usr/local/lib/python3.12/dist-packages/brax/io/mjcf.py:480: UserWarning: Brax System, piplines and environments are not actively being maintained. Please see MJX for a well maintained JAX-based physics engine: https://github.com/google-deepmind/mujoco/tree/main/mjx. For a host of environments that use MJX, see: https://github.com/google-deepmind/mujoco_playground.
  warnings.warn(
JAX backend gpu  devices [CudaDevice(id=0)]
--- reward fingerprint ---------------------------------------
  code         : 5541353
  contact      : (foot_z - 0.014) < 0.001   [radius-corrected]
  clearance    : COST, target foot z = 0.05
  cmd stage 2  : vx[-0.15,+0.35] vy[-0.15,+0.15] wz[-0.50,+0.50]
  terrain      : 1.00   (rough — sim2real robustness)
  dr scale     : 1.00   (default DR) [+torque-headroom +mass/inertia]
  STAIRCASE    : 0.60 of envs (rise 0.08m*level) [tier-2 teacher, needs terrain>0 + --he

## 8. Probe — the real run (~1.3 h)

**First run** grafts from the teacher. **After a dropout**, re-run this cell: it detects the probe's
own checkpoint and RESUMES it (no re-graft, no lost progress).

Per-eval line to watch: `climb`/`climb_max` (net / peak base-z gain — should lift off ~0 and rise),
`swing` (mean swing-foot height above local ground — should climb toward the 0.08 m riser),
`w_slip`/`airT` (contact fix engaging). **5M kill-switch:** if `swing` hasn't moved and `w_slip`
hasn't engaged by ~5M with `v_loss` plateaued, it's doomed — stop and restart clean.

In [7]:
HM_GRAFT = f"{DRIVE}/nova_policy_hm.pkl"
CLIMB    = f"{DRIVE}/nova_climb_v1"        # FRESH dir
import os; assert os.path.exists(HM_GRAFT), "nova_policy_hm.pkl missing — tell me, need a re-graft cell"
!python train.py --heightmap --stair-frac 0.6 --terrain 1.0 --curriculum --flat-frac 0.25 \
    --restore-params-pkl {HM_GRAFT} --ckpt {CLIMB} --timesteps 120_000_000 --dry-run

Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'
/usr/local/lib/python3.12/dist-packages/brax/io/mjcf.py:480: UserWarning: Brax System, piplines and environments are not actively being maintained. Please see MJX for a well maintained JAX-based physics engine: https://github.com/google-deepmind/mujoco/tree/main/mjx. For a host of environments that use MJX, see: https://github.com/google-deepmind/mujoco_playground.
  warnings.warn(
JAX backend gpu  devices [CudaDevice(id=0)]
--- reward fingerprint ---------------------------------------
  code         : 5541353
  contact      : (foot_z - 0.014) < 0.001   [radius-corrected]
  clearance    : COST, target foot z = 0.05
  cmd stage 2  : vx[-0.15,+0.35] vy[-0.15,+0.15] wz[-0.50,+0.50]
  terrain      : 1.00   (rough — sim2real robustness)
  dr scale     : 1.00   (default DR) [+torque-headroom +mass/inertia]
  STAIRCASE    : 0.60 of envs (rise 0.08m*level) [tier-2 teacher, needs terrain>0 + --he

In [ ]:
!python train.py --heightmap --stair-frac 0.6 --terrain 1.0 --curriculum --flat-frac 0.25 \
    --restore-params-pkl {HM_GRAFT} --ckpt {CLIMB} --out {DRIVE}/nova_policy_climb.pkl \
    --timesteps 120_000_000

Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'
/usr/local/lib/python3.12/dist-packages/brax/io/mjcf.py:480: UserWarning: Brax System, piplines and environments are not actively being maintained. Please see MJX for a well maintained JAX-based physics engine: https://github.com/google-deepmind/mujoco/tree/main/mjx. For a host of environments that use MJX, see: https://github.com/google-deepmind/mujoco_playground.
  warnings.warn(
JAX backend gpu  devices [CudaDevice(id=0)]
--- reward fingerprint ---------------------------------------
  code         : 5541353
  contact      : (foot_z - 0.014) < 0.001   [radius-corrected]
  clearance    : COST, target foot z = 0.05
  cmd stage 2  : vx[-0.15,+0.35] vy[-0.15,+0.15] wz[-0.50,+0.50]
  terrain      : 1.00   (rough — sim2real robustness)
  dr scale     : 1.00   (default DR) [+torque-headroom +mass/inertia]
  STAIRCASE    : 0.60 of envs (rise 0.08m*level) [tier-2 teacher, needs terrain>0 + --he

In [8]:
import subprocess
POLICY_CLIMB = f"{DRIVE}/nova_policy_climb.pkl"   # the atomic --out, always current
for tag, lvl in [("s100", 1.0), ("s050", 0.5)]:
    subprocess.run(["python","rollout.py","--policy",POLICY_CLIMB,"--heightmap",
                    "--stair-level",str(lvl),"--vx","0.25","--steps","600",
                    "--out",f"{DRIVE}/climb_check_{tag}.mp4"], check=False)

In [8]:
import glob, os
have_ckpt = any(os.path.isdir(d) and os.path.basename(d).isdigit()
                for d in glob.glob(f"{CKPT}/**/*", recursive=True))
if have_ckpt:
    print(">>> RESUMING the probe's own checkpoint (teacher graft skipped)")
    !python train.py --heightmap --terrain 1.0 --stair-frac 0.6 --flat-frac 0.25 \
        --ckpt {CKPT} --out {POLICY} --timesteps {TIMESTEPS}
else:
    print(">>> FIRST run: grafting from the teacher")
    !python train.py --heightmap --terrain 1.0 --stair-frac 0.6 --flat-frac 0.25 \
        --restore-params-pkl {TEACHER} --ckpt {CKPT} --out {POLICY} --timesteps {TIMESTEPS}

>>> FIRST run: grafting from the teacher
Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'
/usr/local/lib/python3.12/dist-packages/brax/io/mjcf.py:480: UserWarning: Brax System, piplines and environments are not actively being maintained. Please see MJX for a well maintained JAX-based physics engine: https://github.com/google-deepmind/mujoco/tree/main/mjx. For a host of environments that use MJX, see: https://github.com/google-deepmind/mujoco_playground.
  warnings.warn(
JAX backend gpu  devices [CudaDevice(id=0)]
--- reward fingerprint ---------------------------------------
  code         : 5541353
  contact      : (foot_z - 0.014) < 0.001   [radius-corrected]
  clearance    : COST, target foot z = 0.05
  cmd stage 2  : vx[-0.15,+0.35] vy[-0.15,+0.15] wz[-0.50,+0.50]
  terrain      : 1.00   (rough — sim2real robustness)
  dr scale     : 1.00   (default DR) [+torque-headroom +mass/inertia]
  STAIRCASE    : 0.60 of envs (rise 0.08m*level


KeyboardInterrupt



## 9. JUDGE — rollouts + the `climbed` number (the verdict)

`{POLICY}` is written atomically every eval, so it is always the latest policy — the rollouts read
it directly. Videos go to **Drive** (survive disconnect). Read each printed
`traveled +X m in x, climbed +X.XX m in z` line.

- **Stairs** `--stair-level 1.0`: acceptance `climbed >= +0.16 m` (two real 8 cm risers — the
  terrain's TZ=0.20 ceiling caps relief there) and a full 601-frame episode.
- **Flat** `--stair-level 0.0 --vx 0.35`: must survive all 601 frames (today's teacher falls ~470).

In [ ]:
import subprocess, os
for name, level, vx in [("stairs", 1.0, 0.25), ("flat", 0.0, 0.35)]:
    for ext in ("mp4","gif"):
        out = f"{DRIVE}/probe_{name}.{ext}"
        subprocess.run(["python","rollout.py","--policy",POLICY,"--heightmap",
                        "--stair-level",str(level),"--vx",str(vx),
                        "--steps","600","--out",out], check=False)
        if os.path.exists(out):
            print("video:", out); break
        print("failed:", out, "- trying next format")

## 10. (Fallback) rebuild the pkl from the newest checkpoint

Only if you suspect `{POLICY}` is stale. Uses `find_latest_checkpoint` (stdlib, no jax) — the SAME
pointer-based selector `train.py` uses — **not** max-by-step, which picks the wrong run after a
resume (each resume starts a new `run_*/` with brax's step counter reset to 0). Shells out to
`graft_obs.py` so no jax enters the kernel.

In [ ]:
import subprocess
from ckpt_utils import find_latest_checkpoint      # pure stdlib — safe in the kernel
newest = find_latest_checkpoint(CKPT)
assert newest, f"no checkpoint under {CKPT}"
print("newest checkpoint:", newest)
subprocess.run(["python","graft_obs.py","--src",newest,"--add-dims","0","--out",POLICY], check=True)
print("wrote", POLICY)

## 11. Export for deploy (only once the probe passes)

WARNING: this is a **privileged teacher** — obs 226 includes the *perfect* heightmap the real
D456/L2 cannot supply, so the exported `.npz` cannot run on the Jetson as-is. Hardware needs real
elevation mapping or student distillation onto proprioception-only obs. The deployable policy is
still the flat 105-d one.

In [ ]:
!python export_policy.py --policy {POLICY}
!cp -v nova_policy.npz nova_policy.onnx {DRIVE}/ 2>/dev/null; echo "exported to Drive"